In [73]:
import pandas as pd
import sqlite3

<h2>Создайте подключение к базе данных с помощью sqlite3 библиотеки.</h2>

In [74]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

<h2>Создайте в базе данных новую таблицу, названную так datamart, путем объединения таблиц pageviews и checker использования только одного запроса.</h2>

- Таблица должна иметь следующие столбцы: «uid», «labname», «first_commit_ts» и «first_view_ts».
"first_commit_ts" — новое название столбца "timestamp" в таблице проверки. Он показывает первый коммит из конкретной лаборатории и пользователя.
"first_view_ts" показывает время первого посещения пользователем таблицы просмотров страниц. Это временная метка посещения пользователем новостной ленты.

- status = 'ready' все равно должен быть фильтр.

- numTrials = 1 все равно должен быть фильтр.
- «labnames» должно быть из списка: laba04, laba04s, laba05, laba06, laba06s, и project1.
- Таблица должна содержать только пользователей (uid с user_*), а не администраторов.
- «first_commit_ts» и «first_view_ts» следует обрабатывать как datetime64[ns].

In [75]:
conn.execute("DROP TABLE IF EXISTS datamart;")

In [76]:
query = """ 
CREATE TABLE datamart AS
SELECT 
    c.uid, 
    c.labname, 
    MIN(c.timestamp) AS first_commit_ts, 
    MIN(p.datetime) AS first_view_ts
FROM checker AS c
LEFT JOIN pageviews AS p ON c.uid = p.uid
WHERE c.status = 'ready'
  AND c.numTrials = 1
  AND c.labname IN ('laba04', 'laba04s', 'laba05', 'laba06', 'laba06s', 'project1')
  AND c.uid LIKE 'user_%'
GROUP BY c.uid, c.labname;
"""

In [ ]:
conn.execute(query)

In [ ]:
datamart = pd.read_sql("SELECT * FROM datamart;", conn,
                       parse_dates=["first_commit_ts", "first_view_ts"])

In [ ]:
datamart.head()

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136


In [ ]:
datamart.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 140 entries, 0 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              140 non-null    object        
 1   labname          140 non-null    object        
 2   first_commit_ts  140 non-null    datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 4.5+ KB


<h2>Используя методы Pandas, создайте два фрейма данных: test и control.</h2>


- test должны содержать пользователей со значениями в «first_view_ts».

- control должны содержать пользователей с отсутствующими значениями в «first_view_ts».
- Замените отсутствующие значения в контрольной таблице средним значением "first_view_ts" тестовых пользователей. Мы будем использовать это значение для дальнейшего анализа.
- Сохраните обе таблицы в базе данных. Они понадобятся вам в следующих упражнениях.

In [ ]:
test = datamart[datamart["first_view_ts"].notna()].copy()

In [ ]:
test.head(5)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136


In [ ]:
test.info()

<class 'pandas.core.frame.DataFrame'>
Index: 59 entries, 0 to 114
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              59 non-null     object        
 1   labname          59 non-null     object        
 2   first_commit_ts  59 non-null     datetime64[ns]
 3   first_view_ts    59 non-null     datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 2.3+ KB


In [78]:
test = pd.io.sql.read_sql('SELECT * FROM test', conn)

In [ ]:
control = datamart[datamart["first_view_ts"].isna()].copy()

In [ ]:
control.head()

,uid,labname,first_commit_ts,first_view_ts
12,user_11,laba05,2020-05-03 21:06:55.970293,NaT
13,user_11,project1,2020-05-03 23:45:33.673409,NaT
14,user_12,laba04,2020-04-18 17:07:51.767358,NaT
15,user_12,laba04s,2020-04-26 15:42:38.070593,NaT
16,user_12,laba05,2020-05-03 08:39:25.174316,NaT


In [ ]:
control.info()

<class 'pandas.core.frame.DataFrame'>
Index: 81 entries, 12 to 139
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   uid              81 non-null     object        
 1   labname          81 non-null     object        
 2   first_commit_ts  81 non-null     datetime64[ns]
 3   first_view_ts    0 non-null      datetime64[ns]
dtypes: datetime64[ns](2), object(2)
memory usage: 3.2+ KB


In [79]:
control = pd.io.sql.read_sql('SELECT * FROM control', conn)

In [ ]:
mean_ts = test["first_view_ts"].mean()

control["first_view_ts"].fillna(mean_ts, inplace=True)

C:\Users\Анастасия\AppData\Local\Temp\ipykernel_16608\946864009.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  control["first_view_ts"].fillna(mean_ts, inplace=True)


In [ ]:
control.head()

,uid,labname,first_commit_ts,first_view_ts
12,user_11,laba05,2020-05-03 21:06:55.970293,2020-04-27 00:40:05.761783552
13,user_11,project1,2020-05-03 23:45:33.673409,2020-04-27 00:40:05.761783552
14,user_12,laba04,2020-04-18 17:07:51.767358,2020-04-27 00:40:05.761783552
15,user_12,laba04s,2020-04-26 15:42:38.070593,2020-04-27 00:40:05.761783552
16,user_12,laba05,2020-05-03 08:39:25.174316,2020-04-27 00:40:05.761783552


In [ ]:
test.to_sql("test", conn, if_exists="replace", index=False)
control.to_sql("control", conn, if_exists="replace", index=False)

81

<h2>Закройте соединение.</h2>

In [ ]:
conn.close()